# Banking77 Cross-Run Analysis

Comparison of all 8 experiment runs. Read-only — no retraining.

## 1. Setup

In [ ]:
import os, sys
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# set repo root so imports work regardless of launch dir
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.analysis import load_all_runs, load_training_curves, load_per_class_f1

SUMMARY_DIR = 'experiments/_summary'
PLOTS_DIR = os.path.join(SUMMARY_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

RUN_ORDER = [
    'distilbert_baseline', 'distilbert_lr_low', 'distilbert_lr_high',
    'distilbert_batch32', 'distilbert_epochs5',
    'distilbert_frozen_encoder', 'distilbert_frozen_partial', 'bert_baseline',
]
BASELINE = 'distilbert_baseline'
print(f'Repo root: {REPO_ROOT}')

## 2. Load Runs

In [ ]:
df = load_all_runs('experiments')
df = df.sort_values('test_macro_f1', ascending=False).reset_index(drop=True)
display(df[['run_name','model_name','learning_rate','batch_size','num_epochs','freeze_strategy',
            'val_macro_f1','test_acc','test_macro_f1','wall_clock_seconds']])

## 3. Final Metrics Table

In [ ]:
table_cols = ['run_name','model_name','learning_rate','batch_size','num_epochs',
              'freeze_strategy','val_macro_f1','test_acc','test_macro_f1','test_weighted_f1','wall_clock_seconds']
df_table = df[table_cols].copy()

csv_path = os.path.join(SUMMARY_DIR, 'summary_table.csv')
df_table.to_csv(csv_path, index=False)
print(f'Saved {csv_path}')

md_path = os.path.join(SUMMARY_DIR, 'summary_table.md')
md_text = df_table.to_markdown(index=False, floatfmt='.4f')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(md_text)
print(f'Saved {md_path}')
print(md_text)

## 4. Bar Chart: Test Accuracy per Run

In [ ]:
df_bar = df.sort_values('test_acc', ascending=True)
colors = ['steelblue' if r != BASELINE else 'darkorange' for r in df_bar['run_name']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_bar['run_name'], df_bar['test_acc'], color=colors)
ax.set_xlabel('Test Accuracy')
ax.set_title('Test Accuracy per Run (orange = baseline)')
ax.set_xlim(0, 1.0)
for bar, val in zip(bars, df_bar['test_acc']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'test_accuracy_bar.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 5. Bar Chart: Test Macro-F1 per Run

In [ ]:
df_bar = df.sort_values('test_macro_f1', ascending=True)
colors = ['steelblue' if r != BASELINE else 'darkorange' for r in df_bar['run_name']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_bar['run_name'], df_bar['test_macro_f1'], color=colors)
ax.set_xlabel('Test Macro-F1')
ax.set_title('Test Macro-F1 per Run (orange = baseline)')
ax.set_xlim(0, 1.0)
for bar, val in zip(bars, df_bar['test_macro_f1']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'test_macro_f1_bar.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 6. Line Plot: Val Loss Curves Overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for run in RUN_ORDER:
    try:
        curve = load_training_curves(run)
        lw = 2.0 if run == BASELINE else 1.2
        ls = '--' if run == BASELINE else '-'
        ax.plot(curve['epoch'], curve['eval_loss'], label=run, lw=lw, ls=ls, marker='o', ms=4)
    except Exception as e:
        print(f'Skip {run}: {e}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss')
ax.set_title('Validation Loss Curves — All Runs')
ax.legend(fontsize=7, loc='upper right')
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'val_loss_overlay.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 7. Line Plot: Val Macro-F1 Curves Overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for run in RUN_ORDER:
    try:
        curve = load_training_curves(run)
        lw = 2.0 if run == BASELINE else 1.2
        ls = '--' if run == BASELINE else '-'
        ax.plot(curve['epoch'], curve['eval_macro_f1'], label=run, lw=lw, ls=ls, marker='o', ms=4)
    except Exception as e:
        print(f'Skip {run}: {e}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Macro-F1')
ax.set_title('Validation Macro-F1 Curves — All Runs')
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'val_macro_f1_overlay.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 8. Scatter: Wall-Clock vs Test Macro-F1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['wall_clock_seconds'], df['test_macro_f1'], zorder=3)
for _, row in df.iterrows():
    ax.annotate(row['run_name'], (row['wall_clock_seconds'], row['test_macro_f1']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)
ax.set_xlabel('Wall-Clock (seconds)')
ax.set_ylabel('Test Macro-F1')
ax.set_title('Efficiency: Training Time vs Test Macro-F1')
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'time_vs_f1.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 9. Hyperparameter Effect Plots

In [ ]:
# LR sweep
lr_runs = ['distilbert_lr_low', 'distilbert_baseline', 'distilbert_lr_high']
lr_labels = ['2e-5', '5e-5 (base)', '1e-4']
lr_df = df[df['run_name'].isin(lr_runs)].set_index('run_name').reindex(lr_runs)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(lr_labels, lr_df['test_macro_f1'], color=['steelblue', 'darkorange', 'steelblue'])
ax.set_ylabel('Test Macro-F1')
ax.set_title('LR Sweep: Test Macro-F1')
ax.set_ylim(0.7, 1.0)
for bar, val in zip(bars, lr_df['test_macro_f1']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.4f}', ha='center', fontsize=9)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'lr_sweep.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

# Freeze strategy sweep
freeze_runs = ['distilbert_baseline', 'distilbert_frozen_partial', 'distilbert_frozen_encoder']
freeze_labels = ['none (base)', 'partial (3 layers)', 'full encoder']
freeze_df = df[df['run_name'].isin(freeze_runs)].set_index('run_name').reindex(freeze_runs)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(freeze_labels, freeze_df['test_macro_f1'],
              color=['darkorange', 'steelblue', 'firebrick'])
ax.set_ylabel('Test Macro-F1')
ax.set_title('Freeze Strategy: Test Macro-F1')
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, freeze_df['test_macro_f1']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.4f}', ha='center', fontsize=9)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'freeze_sweep.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

# Architecture comparison
arch_runs = ['distilbert_baseline', 'bert_baseline']
arch_labels = ['DistilBERT-base', 'BERT-base']
arch_df = df[df['run_name'].isin(arch_runs)].set_index('run_name').reindex(arch_runs)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(arch_labels, arch_df['test_macro_f1'], color=['steelblue', 'mediumseagreen'])
ax.set_ylabel('Test Macro-F1')
ax.set_title('Architecture: DistilBERT vs BERT-base')
ax.set_ylim(0.85, 1.0)
for bar, val, wc in zip(bars, arch_df['test_macro_f1'], arch_df['wall_clock_seconds']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}\n({wc:.0f}s)', ha='center', fontsize=9)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'arch_comparison.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 10. Per-Class F1 Comparison

In [ ]:
# best DistilBERT run vs frozen_encoder
best_distilbert = df[df['model_name'] == 'distilbert-base-uncased'].iloc[0]['run_name']
run_a = best_distilbert
run_b = 'distilbert_frozen_encoder'

pca = load_per_class_f1(run_a).set_index('class_name')
pcb = load_per_class_f1(run_b).set_index('class_name')
common = pca.index.intersection(pcb.index)
pca_c = pca.loc[common, 'f1'].sort_values(ascending=False)
pcb_c = pcb.loc[pca_c.index, 'f1']

x = list(range(len(pca_c)))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x, pca_c.values, label=f'{run_a}', lw=1.2, marker='.', ms=3)
ax.plot(x, pcb_c.values, label=f'{run_b}', lw=1.2, marker='.', ms=3, alpha=0.7)
ax.set_xticks(x[::5])
ax.set_xticklabels(list(pca_c.index)[::5], rotation=45, ha='right', fontsize=6)
ax.set_ylabel('F1')
ax.set_title(f'Per-Class F1: {run_a} vs {run_b} (sorted by {run_a} desc)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'per_class_comparison.png')
plt.savefig(out, dpi=120)
plt.close()
print(f'Saved {out}')

## 11. Findings

In [ ]:
best = df.iloc[0]
worst = df.iloc[-1]
baseline_row = df[df['run_name'] == 'distilbert_baseline'].iloc[0]
bert_row = df[df['run_name'] == 'bert_baseline'].iloc[0]
epochs5_row = df[df['run_name'] == 'distilbert_epochs5'].iloc[0]
lr_low_row = df[df['run_name'] == 'distilbert_lr_low'].iloc[0]
lr_high_row = df[df['run_name'] == 'distilbert_lr_high'].iloc[0]
batch_row = df[df['run_name'] == 'distilbert_batch32'].iloc[0]
frozen_enc_row = df[df['run_name'] == 'distilbert_frozen_encoder'].iloc[0]
frozen_part_row = df[df['run_name'] == 'distilbert_frozen_partial'].iloc[0]

findings = (
    f'## Key Findings\n\n'
    f'- **Best run overall:** `{best["run_name"]}` test macro-F1 = {best["test_macro_f1"]:.4f}, '
    f'wall-clock {best["wall_clock_seconds"]:.0f}s.\n\n'
    f'- **DistilBERT epochs=5 vs BERT-base:** `distilbert_epochs5` ({epochs5_row["test_macro_f1"]:.4f}) '
    f'outperforms `bert_baseline` ({bert_row["test_macro_f1"]:.4f}) at {epochs5_row["wall_clock_seconds"]:.0f}s vs '
    f'{bert_row["wall_clock_seconds"]:.0f}s \u2014 DistilBERT is the cost-effective choice for Banking77.\n\n'
    f'- **LR sensitivity:** LR=2e-5 degrades to {lr_low_row["test_macro_f1"]:.4f} '
    f'(\u2212{baseline_row["test_macro_f1"]-lr_low_row["test_macro_f1"]:.4f} vs baseline); '
    f'LR=1e-4 improves to {lr_high_row["test_macro_f1"]:.4f} (+{lr_high_row["test_macro_f1"]-baseline_row["test_macro_f1"]:.4f}).\n\n'
    f'- **Frozen encoder collapse:** Head-only training (`distilbert_frozen_encoder`) collapses to '
    f'macro-F1 = {frozen_enc_row["test_macro_f1"]:.4f} \u2014 768-dim CLS embedding is insufficient for 77-way classification '
    f'without fine-tuning the encoder.\n\n'
    f'- **Partial freeze is viable:** Freezing bottom 3 of 6 DistilBERT layers achieves '
    f'{frozen_part_row["test_macro_f1"]:.4f} (only \u2212{baseline_row["test_macro_f1"]-frozen_part_row["test_macro_f1"]:.4f} vs baseline) '
    f'at {frozen_part_row["wall_clock_seconds"]:.0f}s \u2014 useful when GPU budget is tight.\n\n'
    f'- **Batch=32 slightly hurts:** Doubling batch size drops to {batch_row["test_macro_f1"]:.4f} '
    f'(\u2212{baseline_row["test_macro_f1"]-batch_row["test_macro_f1"]:.4f}), likely due to effective LR being too low with warmup_steps=500.\n\n'
    f'- **Val/test gap:** All runs show near-zero generalisation gap. '
    f'Baseline val {baseline_row["val_macro_f1"]:.4f} \u2192 test {baseline_row["test_macro_f1"]:.4f}. No overfitting detected in 3-epoch runs.'
)
print(findings)